# Домашняя работа 3



## Подготовка

In [1]:
from pathlib import Path
import pandas as pd
from pandas.testing import assert_frame_equal
DATA = Path("../data")
WORK = Path("../results")
WORK.mkdir(exist_ok=True)
decisions = pd.read_csv(
    DATA / "decisions.csv", dtype={"lei": "string", "county_code": "string"}
)
print("pandas", pd.__version__, decisions.shape)

pandas 3.1.1 (6830, 7)


## Исходники домашней работы

In [2]:
org_keys = ["lei", "activity_year"]
long_keys = org_keys + ["action_taken"]
organizations = pd.read_csv(DATA / "organizations.csv", dtype={"lei": "string"})
lookup = organizations[org_keys + ["organization_name"]].copy()
assert not lookup.duplicated(org_keys).any()
print(organizations.head(3))

                    lei                       organization_name  source_count  \
0  549300AG64NHILB7ZP05                      LOANDEPOT.COM, LLC           114   
1  QFROUN1UWUYU0DVIWD51  Fifth Third Bank, National Association             3   
2  5493003GQDUH26DNNH17               NAVY FEDERAL CREDIT UNION           742   

   activity_year  
0           2023  
1           2023  
2           2023  


## H1 Профиль отказов

In [5]:
denial_data = decisions.copy()

denial_data["is_denied"] = denial_data["action_taken"].eq(3)

home_denial_profile = (denial_data.groupby(org_keys).agg(n_decisions=("action_taken", "size"),n_denied=("is_denied", "sum")).reset_index())

home_denial_profile["denial_rate"] = (home_denial_profile["n_denied"] /home_denial_profile["n_decisions"])

home_denial_profile = home_denial_profile.merge(lookup,on=org_keys,how="left",validate="one_to_one",indicator=True)

print("Результат соединения:")
print(home_denial_profile["_merge"].value_counts())

assert home_denial_profile["n_decisions"].sum() == len(decisions)
assert home_denial_profile["n_denied"].sum() == denial_data["is_denied"].sum()

assert home_denial_profile["denial_rate"].between(0, 1).all()
home_denial_profile.to_csv(WORK / "home_denial_profile.csv",index=False)

print("\nПервые 5 строк профиля отказов:")
display(home_denial_profile.head())

print("\nРазмер итоговой таблицы:", home_denial_profile.shape)
print("Общее количество решений:", home_denial_profile["n_decisions"].sum())
print("Общее количество отказов:", home_denial_profile["n_denied"].sum())
print("Общая доля отказов:",home_denial_profile["n_denied"].sum() / home_denial_profile["n_decisions"].sum())


Результат соединения:
_merge
both          314
left_only       0
right_only      0
Name: count, dtype: int64

Первые 5 строк профиля отказов:


,lei,activity_year,n_decisions,n_denied,denial_rate,organization_name,_merge
0,03D0JEWFDFUS0SEEKG89,2023,91,15,0.164835,"TD Bank, National Association",both
1,1IE8VN30JCEQV1H4R804,2023,7,2,0.285714,"HSBC BANK USA, NATIONAL ASSOCIATION",both
2,213800KVD76WKI6PN868,2023,2,1,0.500000,"CAPITAL CENTER, L.L.C.",both
3,213800QUAI2VH5YM6310,2023,89,9,0.101124,"EMBRACE HOME LOANS, INC.",both
4,213800XR2TCBQJSF1X93,2023,15,3,0.200000,NBKC BANK,both



Размер итоговой таблицы: (314, 7)
Общее количество решений: 6830
Общее количество отказов: 691
Общая доля отказов: 0.10117130307467057


## H2 Группы для просмотра

In [6]:
n_groups_before = len(home_denial_profile)

large_groups = home_denial_profile.loc[home_denial_profile["n_decisions"].ge(20)].copy()

large_groups.to_csv(WORK / "home_denial_groups20.csv",index=False)

overall_denial_rate = (home_denial_profile["n_denied"].sum() / home_denial_profile["n_decisions"].sum())

print("Количество групп до фильтрации:", n_groups_before)
print("Количество групп после фильтрации:", len(large_groups))

print("\nОбщая доля отказов по всем организациям:", round(overall_denial_rate * 100, 2), "%")

print("\nПервые 5 крупных групп:")
display(large_groups.head())

print("\nРазмер итоговой таблицы:", large_groups.shape)


Количество групп до фильтрации: 314
Количество групп после фильтрации: 61

Общая доля отказов по всем организациям: 10.12 %

Первые 5 крупных групп:


,lei,activity_year,n_decisions,n_denied,denial_rate,organization_name,_merge
0,03D0JEWFDFUS0SEEKG89,2023,91,15,0.164835,"TD Bank, National Association",both
3,213800QUAI2VH5YM6310,2023,89,9,0.101124,"EMBRACE HOME LOANS, INC.",both
9,2549003Z8R5QM0WPS603,2023,24,0,0.000000,IDB Global Federal Credit Union,both
14,2549006II76YXSS5XM65,2023,193,7,0.036269,"George Mason Mortgage, LLC",both
26,254900FMBXF85WQYV433,2023,20,1,0.050000,Amalgamated Bank,both



Размер итоговой таблицы: (61, 7)


## H3 Длинная, широкая и снова длинная форма

In [9]:
long_counts = decisions.groupby(long_keys).size().rename("n").reset_index()

wide_counts = long_counts.pivot(index=org_keys, columns="action_taken", values="n").reindex(columns=[1, 2, 3], fill_value=0).fillna(0).astype("int64")

long_restored = wide_counts.reset_index().melt(id_vars=org_keys, var_name="action_taken", value_name="n")

long_restored["action_taken"] = long_restored["action_taken"].astype("int64")
long_restored["n"] = long_restored["n"].astype("int64")

long_restored = long_restored.loc[long_restored["n"] > 0].copy()

long_counts = long_counts.sort_values(long_keys).reset_index(drop=True)
long_restored = long_restored.sort_values(long_keys).reset_index(drop=True)

assert_frame_equal(long_counts, long_restored)

print("Проверка пройдена! Таблицы полностью совпадают.")
display(long_counts.head())
display(wide_counts.head())
display(long_restored.head())

print("Размер первоначальной таблицы:", long_counts.shape)
print("Размер восстановленной таблицы:", long_restored.shape)


Проверка пройдена! Таблицы полностью совпадают.


,lei,activity_year,action_taken,n
0,03D0JEWFDFUS0SEEKG89,2023,1,76
1,03D0JEWFDFUS0SEEKG89,2023,3,15
2,1IE8VN30JCEQV1H4R804,2023,1,5
3,1IE8VN30JCEQV1H4R804,2023,3,2
4,213800KVD76WKI6PN868,2023,1,1


,action_taken,1,2,3
lei,activity_year,,,
03D0JEWFDFUS0SEEKG89,2023,76,0,15
1IE8VN30JCEQV1H4R804,2023,5,0,2
213800KVD76WKI6PN868,2023,1,0,1
213800QUAI2VH5YM6310,2023,76,4,9
213800XR2TCBQJSF1X93,2023,12,0,3


,lei,activity_year,action_taken,n
0,03D0JEWFDFUS0SEEKG89,2023,1,76
1,03D0JEWFDFUS0SEEKG89,2023,3,15
2,1IE8VN30JCEQV1H4R804,2023,1,5
3,1IE8VN30JCEQV1H4R804,2023,3,2
4,213800KVD76WKI6PN868,2023,1,1


Размер первоначальной таблицы: (500, 4)
Размер восстановленной таблицы: (500, 4)


## H+ Безопасное соединение